### Segment-Wise ISC Analysis

In [1]:
import numpy as np
import pandas as pd
import os, glob
import utils.segment_isc_utils # Assuming your module works as before

### Set up Segments and Data

In [2]:
# --- CONFIGURATION ---
DATA_DIR        = "../data/02_processed_combined_data"
CONDITIONS_FILE = "../data/sub_conditions.csv"
OUTPUT_DIR      = "../data/04_analysis/h2_segment_results/all_subjects/"

if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

SEGMENTS = {
    'PC':  {1: (9, 54), 2: (55, 205), 3: (205, 243), 4: (243, 278), 5: (279, 319)},
    'PCM': {1: (9, 54), 2: (55, 205), 3: (205, 240), 4: (240, 278), 5: (279, 319)}
}

signals = ['Gaze_X', 'Gaze_Y', 'Pupil_Mean', 'Heart_Rate', 'GSR']


### Define Helper Function and Run

In [3]:
# --- HELPER: PERMUTATION STATS ---
def calculate_permuted_stats(df_a, df_b, n_permutations=5000):
    """
    Returns: Mean r, Low_Err (distance), High_Err (distance)
    """
    n_a, n_b = df_a.shape[1], df_b.shape[1]
    k_a, k_b = int(n_a // 2), int(n_b // 2)
    
    if k_a < 1 or k_b < 1: return 0, 0, 0
    
    r_values = np.zeros(n_permutations)
    
    for i in range(n_permutations):
        idx_a = np.random.choice(n_a, k_a, replace=False)
        idx_b = np.random.choice(n_b, k_b, replace=False)
        g1 = df_a.iloc[:, idx_a].mean(axis=1)
        g2 = df_b.iloc[:, idx_b].mean(axis=1)
        r = np.corrcoef(g1, g2)[0, 1]
        r_values[i] = np.clip(r, -0.999999, 0.999999)

    # Fisher Z Statistics
    z_values = np.arctanh(r_values)
    mean_z = np.mean(z_values)
    ci_lower_z = np.percentile(z_values, 2.5)
    ci_upper_z = np.percentile(z_values, 97.5)
    
    mean_r = np.tanh(mean_z)
    # Return distances for error bars
    return mean_r, (mean_r - np.tanh(ci_lower_z)), (np.tanh(ci_upper_z) - mean_r)

def load_physio_data(data_dir, subject_ids, signal_type='GSR'):
    """
    Loads physiological data for a list of subjects.
    
    Args:
        data_dir (str): Path to processed combined data.
        subject_ids (list): List of subject IDs to load (e.g. ['sub001', 'sub002']).
        signal_type (str): 'GSR' or 'Heart_Rate'.
        
    Returns:
        pd.DataFrame: DataFrame with shape (n_timepoints, n_subjects).
                      Columns are subject IDs.
    """
    aligned_data = {}
    valid_subjects = []
    
    # Find all files first to handle case insensitivity (sub vs Sub)
    # We look for both '_combined_physio.csv' AND '_processed_et.csv' or similar depending on where ET is.
    # Actually, previous conversation said "Integrate Physiological Data" into combined csv.
    # Let's check if Gaze/Pupil columns are in the SAME combined file or different files.
    # The 'sub001_combined_physio.csv' we viewed earlier HAD 'Pupil_Mean', 'Gaze_X', 'Gaze_Y'.
    # So we just need to ensure we map those column names correctly.
    
    all_files = glob.glob(os.path.join(data_dir, '*combined_physio.csv'))
    file_map = {}
    for f in all_files:
        basename = os.path.basename(f)
        file_id = basename.split('_')[0].lower()
        file_map[file_id] = f
        
    lengths = []
    
    for sub_id in subject_ids:
        # Check if we have a file for this subject
        lookup_id = sub_id.lower()
        
        if lookup_id not in file_map:
            #print(f"  Warning: No data file found for subject {sub_id}")
            continue
            
        file_path = file_map[lookup_id]
        
        try:
            df = pd.read_csv(file_path)
            
            # Column Mapping fallback
            col_map = {
                'GSR': ['GSR', 'GSR_Interpolated'],
                'Heart_Rate': ['Heart_Rate', 'Heart Rate'],
                'Gaze_X': ['Gaze_X', 'Interpolated Gaze X', 'X'],
                'Gaze_Y': ['Gaze_Y', 'Interpolated Gaze Y', 'Y'],
                'Pupil_Mean': ['Pupil_Mean', 'Mean_Pupil', 'Pupil Diameter']
            }
            
            target_col = signal_type
            if signal_type in col_map:
                for possible in col_map[signal_type]:
                    if possible in df.columns:
                        target_col = possible
                        break
            
            if target_col not in df.columns:
                print(f"  Warning: Column '{signal_type}' (or variants) not found in {os.path.basename(file_path)}")
                continue
            
            signal_data = df[target_col].values
            
            # Basic cleaning: handle infinite or NaN
            if not np.isfinite(signal_data).all():
                # Interpolate linear
                signal_series = pd.Series(signal_data)
                # First interpolate internal holes
                signal_series = signal_series.interpolate(method='linear', limit_direction='both')
                # Then fill edges just in case
                signal_series = signal_series.ffill().bfill()
                signal_data = signal_series.values
                
            # If still NaNs (e.g. all NaNs), warning or skip
            if np.isnan(signal_data).any():
                #print(f"  Warning: {sub_id} {target_col} contains NaNs after interpolation. Dropping.")
                continue
                
            aligned_data[sub_id] = signal_data
            valid_subjects.append(sub_id)
            lengths.append(len(signal_data))
            
        except Exception as e:
            print(f"  Error loading {sub_id}: {e}")
            
    if not aligned_data:
        raise ValueError("No valid data loaded for any subject!")
        
    # Determine common length
    # We use the minimum length to ensure no NaNs at the end, 
    # assuming start times are aligned (which they should be from previous processing).
    # However, if min length is too short (bad subject), we might want to exclude that subject.
    # Let's verify lengths.
    
    # Histogram of lengths
    length_counts = pd.Series(lengths).value_counts().sort_index()
    #print(f"\nData Length Distribution for {signal_type}:")
    #print(length_counts)
    
    # Use the most common length or min length?
    # If we use min, we might cut good data because of one short file.
    # If we use mode, we might need to pad or trim.
    # Let's use the minimum length that is "reasonable" (e.g. at least 90% of median).
    median_len = np.median(lengths)
    threshold = 0.9 * median_len
    
    # Filter out subjects that are too short
    final_subjects = []
    clean_data_dict = {}
    
    valid_lengths = [l for l in lengths if l >= threshold]
    if not valid_lengths:
         raise ValueError("All loaded files are too short compared to median length!")
         
    target_len = min(valid_lengths) # Trim everyone to the shortest VALID length
    #print(f"Trimming data to common length: {target_len} samples.")
    
    skipped_short = 0
    for sub in valid_subjects:
        data = aligned_data[sub]
        if len(data) < target_len:
            # This subject is too short (below threshold), skip
            # print(f"  Dropping {sub}: length {len(data)} < {target_len}")
            skipped_short += 1
            continue
        
        # Trim to target_len
        clean_data_dict[sub] = data[:target_len]
        final_subjects.append(sub)
        
    if skipped_short > 0:
        print(f"Dropped {skipped_short} subjects due to insufficient data length.")
        
    #print(f"Successfully loaded {len(final_subjects)} subjects for {signal_type}.\n")
    
    # Create DataFrame
    df_final = pd.DataFrame(clean_data_dict)
    
    return df_final

def get_subjects_by_condition(conditions_file, target_condition='PC', exclude_prior_exposure=False):
    """
    Reads the conditions file and returns a list of subject IDs for the target condition.
    
    Parameters:
    - exclude_prior_exposure (bool): If True, removes subjects where 'Seen' == 1.
    """
    if not os.path.exists(conditions_file):
        raise FileNotFoundError(f"Conditions file not found: {conditions_file}")
        
    df = pd.read_csv(conditions_file)
    
    # Clean column names
    df.columns = [c.strip() for c in df.columns]
    
    # Check required columns
    required = ['RespondentName', 'SourceStimulus']
    if not set(required).issubset(df.columns):
        raise ValueError(f"Conditions file must contain {required} columns.")
    
    # 1. Filter by Target Condition first
    condition_df = df[df['SourceStimulus'].str.strip() == target_condition].copy()
    
    # 2. Filter by Prior Exposure (New Logic)
    if exclude_prior_exposure:
        if 'Seen' in condition_df.columns:
            # Get initial count for reporting
            n_before = len(condition_df)
            
            # Filter: Keep rows where Seen is NOT 1 (handles 0, NaN, or empty)
            # We convert to numeric to be safe against "1" (str) vs 1 (int)
            condition_df['Seen'] = pd.to_numeric(condition_df['Seen'], errors='coerce').fillna(0)
            condition_df = condition_df[condition_df['Seen'] != 1]
            
            n_excluded = n_before - len(condition_df)
            if n_excluded > 0:
                print(f"[{target_condition}] Excluded {n_excluded} subjects due to prior exposure.")
        else:
            print(f"Warning: 'exclude_prior_exposure' was True, but 'Seen' column was not found in {conditions_file}.")

    # 3. Extract IDs
    subject_ids = condition_df['RespondentName'].apply(parse_subject_id).tolist()
    
    # Remove duplicates and sort
    subject_ids = sorted(list(set(subject_ids)))
    
    return subject_ids


def parse_subject_id(respondent_name):
    """
    Extracts subject ID (e.g., 'sub001') from RespondentName string.
    Handles potential extra spaces or underscores.
    """
    # Split by underscore and take the first part
    parts = respondent_name.split('_')
    sub_id = parts[0].strip()
    return sub_id

### Main Loop Analysis

In [4]:
pc_subs  = get_subjects_by_condition(CONDITIONS_FILE, target_condition='PC', exclude_prior_exposure=False)
pcm_subs = get_subjects_by_condition(CONDITIONS_FILE, target_condition='PCM', exclude_prior_exposure=False)

# --- MAIN LOOP ---
master_results = []
print("Starting Segment-Wise Analysis...")

for signal in signals:
    print(f"  Processing {signal}...")
    try:
        # Load Data
        df_pc  = load_physio_data(DATA_DIR, pc_subs, signal_type=signal)
        df_pcm = load_physio_data(DATA_DIR, pcm_subs, signal_type=signal)
        
        # Trim to common length
        L = min(df_pc.shape[0], df_pcm.shape[0])
        df_pc, df_pcm = df_pc.iloc[:L], df_pcm.iloc[:L]
        
        # Extract Segments
        segs_pc  = utils.segment_isc_utils.extract_segments(df_pc, SEGMENTS['PC'], L)
        segs_pcm = utils.segment_isc_utils.extract_segments(df_pcm, SEGMENTS['PCM'], L)

        # 1. COMMON (Segs 1, 2, 5)
        d_com_pc  = pd.concat([segs_pc[1], segs_pc[2], segs_pc[5]], axis=0)
        d_com_pcm = pd.concat([segs_pcm[1], segs_pcm[2], segs_pcm[5]], axis=0)
        
        # 2. SCRAMBLED (Segs 3, 4 - Time Aligned / Content Mismatch)
        d_scr_pc  = pd.concat([segs_pc[3], segs_pc[4]], axis=0)
        d_scr_pcm = pd.concat([segs_pcm[3], segs_pcm[4]], axis=0)
        # Trim scrambled to min length
        min_s = min(d_scr_pc.shape[0], d_scr_pcm.shape[0])
        d_scr_pc, d_scr_pcm = d_scr_pc.iloc[:min_s], d_scr_pcm.iloc[:min_s]
        
        # 3. UNSCRAMBLED (Segs 3, 4 - Content Aligned / Time Mismatch)
        # PC: 3 then 4. PCM: 4 then 3. We reorder PCM to match PC.
        d_un_pc  = pd.concat([segs_pc[3], segs_pc[4]], axis=0)
        d_un_pcm = pd.concat([segs_pcm[4], segs_pcm[3]], axis=0) # SWAPPED
        # Trim unscrambled
        min_u = min(d_un_pc.shape[0], d_un_pcm.shape[0])
        d_un_pc, d_un_pcm = d_un_pc.iloc[:min_u], d_un_pcm.iloc[:min_u]
        
        # Calculate Stats
        tasks = [("Common", d_com_pc, d_com_pcm),
                 ("Scrambled", d_scr_pc, d_scr_pcm),
                 ("Unscrambled", d_un_pc, d_un_pcm)]
        
        for cond, da, db in tasks:
            m, el, eh = calculate_permuted_stats(da, db)
            master_results.append({
                "Signal": signal, "Condition": cond, 
                "Mean_r": m, "Err_Low": el, "Err_High": eh
            })
            
    except Exception as e:
        print(f"    Error {signal}: {e}")

# Save
pd.DataFrame(master_results).to_csv(os.path.join(OUTPUT_DIR, "h2_segment_stats.csv"), index=False)
print("Computations Complete.")

Starting Segment-Wise Analysis...
  Processing Gaze_X...
  Processing Gaze_Y...
  Processing Pupil_Mean...
  Processing Heart_Rate...
  Processing GSR...
Computations Complete.
